Propósito: Filtra los tickers propuestos para traerlos filtrados por condiciones.

Usa parametros del indicador RSI de 60min, 1d y 7d.

Este filtro busca capturar movimientos de la siguiente forma: Activo sobrevendido en 1d y 60m; la temporalidad baja indica el movimiento de reversion alcista en conjunto con una temporalidad diaria subiendo. La temporalidad más larga indica la tendencia del activo en largo plazo.

In [44]:
# Instala las librerias requeridas.

%pip install -q pandas-ta yfinance lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
# Importa las librerias requeridas.
import pandas as pd
import requests
import io
import yfinance as yf
import pandas_ta as ta

In [46]:
#Scrapea Wikipedia para acceder a los nombres de los tickers del S&P500.

#1.1 Define la URL y un 'User-Agent' para evitar el bloqueo.
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

#1.2 Realiza la petición con el header de seguridad
r = requests.get(url, headers=header)

#1.3 Lee la tabla desde el texto de la respuesta, usando io.StringIO para forzar la interpretación como string HTML
tables = pd.read_html(io.StringIO(r.text))
df_spy = tables[0]

#1.4 Extrae la lista de Tickers
tickers_a_analizar = [t.replace('.', '-') for t in df_spy['Symbol'].tolist()]

#1.5 Añade tickers de interes.

mis_tickers = ['^VIX', 'SPY', 'SPXL' , 'TQQQ' , 'QQQ' , 'URA' , 'XLI' , 'XLF' , 'XLRE' , 'XLB' , 'XLY' , 'XLE', 'XLV' ,
                'XBI' , 'XLC' , 'BTC-USD' , 'DIA']

tickers_a_analizar.extend(mis_tickers)

if tickers_a_analizar:
  print(f"Se han cargado {len(tickers_a_analizar)} tickers...")

Se han cargado 520 tickers...


In [47]:
#DESCARGA DE DATOS TEMPORALIDAD 60 MINUTOS.

tickers_limpios = tickers_a_analizar
data_60m = yf.download(tickers_limpios, period="2mo", interval="60m", group_by='ticker', progress=False)

In [48]:
#ANALISIS TEMPORALIDAD 60MIN.

empresas_60min = []

for ticker in tickers_limpios:
    try:
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables para validación.
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]

        # Lógica de Filtrado 60min
        if (rsi_act <= 35 
            and rsi_act >= rsi_ant
                ):
            
            empresas_60min.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                #Añade el MACD.
                'MACD': round(macd_act, 3)
                
            })
    except Exception as e:
      print(f"Error en {ticker}: {e}")

print("\n" + "="*95)
if empresas_60min:
    df_final = pd.DataFrame(empresas_60min)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales detectadas.")


Ticker  RSI actual  RSI anterior    MACD
   CEG        22.2          19.9  -5.160
    GE        22.2          19.1  -2.316
   DVA        24.3          22.0  -0.494
  META        26.3          25.9 -10.616
  UBER        26.9          24.5  -0.339
    PH        28.0          27.2  -2.657
    RL        28.9          24.7  -1.257
  CSCO        29.1          28.4  -1.431
     F        30.2          28.7  -0.038
    BG        30.3          29.4  -1.086
  PANW        31.7          28.3  -8.643
  CRWD        32.8          28.9  -5.441
   GPC        32.8          24.5  -1.024
   ADM        33.7          30.3  -0.562


In [49]:
# DESCARGA DATOS DIARIOS
data_1d = yf.download(tickers_limpios, period="2mo", interval="1d", group_by='ticker', progress=False)

In [50]:
# ANÁLISIS DIARIO

empresas_diario = []

for ticker in tickers_limpios:
    try:
        df = data_1d[ticker].dropna()
        if len(df) < 30: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=14, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_ant = df['RSI_14'].iloc[-2]
        rsi_act = df['RSI_14'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        

        precio_act = df['Close'].iloc[-1]

        # LÓGICA DE FILTRADO DIARIO
        estuvo_sobrevendido = rsi_act <= 35 or rsi_ant <= 35
        dejo_de_caer = rsi_act >= rsi_ant

        if dejo_de_caer:
            
            empresas_diario.append({
                'Ticker': ticker,
                #'Prec diario': round(precio_act, 2),
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3),
                #'MACD_SIGNAL': round(sig_act , 3)
            })
    except:
        continue

print("\n" + "="*95)


print("PAPELES DETECTADOS EN DIARIO:")
if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales diarias.")


PAPELES DETECTADOS EN DIARIO:
 Ticker  RSI actual  RSI anterior     MACD
    MGM        23.1          21.0   -2.716
    BAC        26.0          20.3   -2.229
   FLEX        27.8          27.6   -4.894
    MOS        29.6          29.1   -1.527
   SKYD        30.2          25.9   -0.876
    ARE        31.4          25.1   -2.437
   SMCI        32.3          32.3   -1.531
   COHR        32.7          32.2  -13.574
    STX        33.2          30.3  -47.542
   SCHW        33.6          29.4   -3.732
     BG        33.7          33.5   -2.798
   AKAM        34.3          29.9   -4.795
   INTC        35.0          33.2   -2.260
    WDC        35.0          32.1  -25.992
   SNDK        35.3          34.3  -70.414
    GLW        35.4          35.4   -4.906
    TER        35.9          35.1   -8.920
   EQIX        36.4          32.7  -13.206
    STT        36.6          31.6   -4.981
   DELL        37.5          34.9   -9.692
     MS        37.7          33.5   -6.977
     MU        37.7    

In [51]:
#TICKERS REPETIDOS EN AMBAS TEMPORALIDADES.

tickers_60 = [e['Ticker'] for e in empresas_60min]
ticker_1d = [e['Ticker'] for e in empresas_diario]
tickers_comunes = list(set(tickers_60) & set(ticker_1d)) # Interseccion de los elementos en la lista.

print(tickers_comunes)


['BG', 'ADM']


In [52]:
# DESCARGA DATOS SEMANAL.
data_7d = yf.download(tickers_limpios, period="1y", interval="1wk", group_by='ticker', progress=False)

In [53]:
# ANÁLISIS SEMANAL

empresas_semanal = []

for ticker in tickers_limpios:
    try:
        df = data_7d[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=10, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_act = df['RSI_10'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]

        # Lógica de filtrado.
        if rsi_act <= 44 and (rsi_act > rsi_ant):
        
            empresas_semanal.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3),
            })
    except:
        continue

print("\n" + "="*95)
print("ACTIVOS DETECTADOS EN SEMANAL.")
if empresas_semanal:
    df_final = pd.DataFrame(empresas_semanal)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales semanales.")


ACTIVOS DETECTADOS EN SEMANAL.
Ticker  RSI actual  RSI anterior     MACD
   MGM        16.5          16.1   -2.806
  WYNN        26.0          21.3   -8.032
   URA        27.5          27.4   -2.637
   ALB        27.8          27.1  -13.744
  SKYD        28.4          26.1   -0.721
    GS        28.8          27.2  -11.989
   PLD        29.3          28.1   -2.073
   LII        30.5          28.8  -40.039
   APP        31.9          28.9  -60.723
   PCG        32.2          30.5   -1.014
  APTV        32.3          31.7   -6.112
   CRH        32.7          29.3   -7.290
   ARE        32.9          32.6   -1.207
    BA        33.2          32.1   -7.495
   CMI        34.5          32.9  -20.360
  AKAM        34.7          32.7   -3.958
   MAA        35.4          34.0   -3.420
  LULU        35.7          32.8  -13.553
   TXT        35.9          34.5   -3.804
  AXON        36.2          34.7  -25.397
  JBHT        36.4          36.4   -2.943
  FISV        36.7          34.6   -3.194
  

In [54]:
#TICKERS REPETIDOS EN TRIPLE PANTALLA.

tickers_60 = [e['Ticker'] for e in empresas_60min]
tickers_1d = [e['Ticker'] for e in empresas_diario]
#tickers_7d = [e['Ticker'] for e in empresas_semanal]

tickers_comunes = list(set(tickers_60) & set(tickers_1d) 
                       #& set(tickers_7d)
                       )

if tickers_comunes:
    print(tickers_comunes)

['BG', 'ADM']
